In [1]:
%load_ext autoreload
%autoreload 2

# Phase 1

In [ ]:
import glob
import os
import sys
import gymnasium as gym
import torch

# Ensure repository root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.env_adapter import MatchEnv
from src.rl_transformer.pool import PoolOpponentController
from src.rl_transformer.ppo import train_mappo
from src.rl_transformer.transformer_model import TransformerActorCritic

# ── Hardware Performance Flags ──
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Bootstrapping Arena Dimensions & Physics ──
TEAM_SIZE = 2
PITCH_W_REG = 840.0
PITCH_H_REG = 500.0
GOAL_H_REG = 450.0   # Large goals accelerate initial goal-discovery
ROUND_STEPS = 900    # 15-second fast rounds
ACTION_REPEAT = 6
NUM_ENVS = 24

# ── Directories & Scratch Pool Initialization ──
SAVE_DIR_S2_P1 = "models/stage2/phase1"
POOL_DIR_S2_P1 = os.path.join(SAVE_DIR_S2_P1, "pool")
os.makedirs(POOL_DIR_S2_P1, exist_ok=True)

# 1. Initialize fresh random model
model_s2_p1 = TransformerActorCritic().to(device)

# 2. Clear out any stale history files in this pool directory
for old_pt in glob.glob(os.path.join(POOL_DIR_S2_P1, "*.pt")):
  os.remove(old_pt)

# 3. Seed pool directly with initial random weights (fair self-play baseline)
torch.save(model_s2_p1.state_dict(), os.path.join(POOL_DIR_S2_P1, "champion.pt"))
torch.save(model_s2_p1.state_dict(), os.path.join(POOL_DIR_S2_P1, "history_0.pt"))
print("🌱 Stage 2 2v2 pool seeded with pure random weights (training from scratch).")


# ── Environment Factory ──
def make_s2_p1_env(env_rank: int):
  def _thunk():
    torch.set_num_threads(1)

    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S2_P1,
        team="blue",
        device="cpu",
        p_random=0.40,      # 40% Random Bot
        p_heuristic=0.0,    # 0% Heuristic
        frame_stack=3,      # Remaining 60% Self-Play
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
        random_reset_opponents=["random"],
        frame_stack=3,
    )
    env.reset(seed=6000 + env_rank)
    return env

  return _thunk


envs_s2_p1 = gym.vector.AsyncVectorEnv(
    [make_s2_p1_env(i) for i in range(NUM_ENVS)],
    context="fork",
    shared_memory=False,
)

# ── Training Loop from Scratch ──
train_mappo(
    envs=envs_s2_p1,
    model=model_s2_p1,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=15_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    update_epochs=3,
    minibatch_size=1024,
    lr_init=5e-4,           # 3e-4 prevents attention LayerNorm divergence
    lr_final=1e-5,
    ent_coef_init=0.025,    # High initial exploration for motor primitives
    ent_coef_final=0.001,
    gamma=0.99,
    gae_lambda=0.95,
    active_tiers=["random", "champion"],
    target_tier="champion",
    filter_thresholds={"random": 0.80},  # Accessible filter for early scratch runs
    tier_ratios={"random": 0.50, "champion": 0.50},
    eval_episodes=100,
    eval_freq=250_000,
    save_dir=SAVE_DIR_S2_P1,
    pool_dir=POOL_DIR_S2_P1,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s2_p1.close()

Using device: cuda
🌱 Stage 2 2v2 pool seeded with pure random weights (training from scratch).
🚀 Entity-Transformer MAPPO Initialized | Format: 2v2 | Envs: 24 | Batch: 12288 | Device: cuda

📊 [EVALUATION @ Step 258,048 | Rollout SPS: 3121 | Tiers: ['random', 'champion']]
   ⚔️  vs Random    [FILTER] | WR:  46.0% | Reward: +1.312 | Goals: 52 Scored, 1 Conceded (+51 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: None, Reward: None, Net: None] (Eval took 4.7s)

📊 [EVALUATION @ Step 503,808 | Rollout SPS: 3129 | Tiers: ['random', 'champion']]
   ⚔️  vs Random    [FILTER] | WR:  94.0% | Reward: +4.694 | Goals: 172 Scored, 0 Conceded (+172 Net)
   ⚔️  vs Champion  [TARGET] | WR:  96.0% | Reward: +6.968 | Goals: 276 Scored, 2 Conceded (+274 Net)
🏆 NEW CHAMPION REGISTERED @ step 503,808 -> models/stage2/phase1_scratch/pool/history_503808.pt
   ⭐⭐ PROMOTED! New Best Score (champion) -> [WR: 96.0%, Reward: +6.968, Net: +274]
      (Defeated previous record: [WR: 9

In [ ]:
import os
import torch
from src.rl_transformer.visualization import evaluate_and_generate_html

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

html_path = evaluate_and_generate_html(
    red_agent="models/stage2/phase1/best_model.pt",
    blue_agent="models/stage2/phase1/final_model.pt",
    red_team_size=2,
    blue_team_size=2,
    device=device,
    filename="stage2_selfplay_test.html",
    num_episodes=10,
    max_steps=900,
    action_repeat=10,
    pitch_width=1200.0,
    pitch_height=800.0,
    goal_height=220.0,
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/Notebooks/training_transformer/render/stage2_heuristic_test.html


# Phase 2

In [9]:
import os
import shutil
import sys
import gymnasium as gym
import torch

# Ensure repository root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.env_adapter import MatchEnv
from src.rl_transformer.pool import PoolOpponentController
from src.rl_transformer.ppo import train_mappo
from src.rl_transformer.transformer_model import TransformerActorCritic

# ── Hardware Performance Flags ──
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Regulation Dimensions & Physics ──
TEAM_SIZE = 2
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS = 3600
ACTION_REPEAT = 6
NUM_ENVS = 24

# ── Directories & Seed Checkpoints ──
SAVE_DIR_S2_P1 = "models/stage2/phase1"
SAVE_DIR_S2_P2 = "models/stage2/phase2"
POOL_DIR_S2_P2 = os.path.join(SAVE_DIR_S2_P2, "pool")

os.makedirs(SAVE_DIR_S2_P2, exist_ok=True)
os.makedirs(POOL_DIR_S2_P2, exist_ok=True)

seed_file = os.path.join(SAVE_DIR_S2_P1, "best_model.pt")
if not os.path.exists(seed_file):
  seed_file = os.path.join(SAVE_DIR_S2_P1, "final_model.pt")

if not os.path.exists(seed_file):
  raise FileNotFoundError(f"Missing Stage 2 Phase 1 checkpoint: {seed_file}")

shutil.copy(seed_file, os.path.join(POOL_DIR_S2_P2, "champion.pt"))
shutil.copy(seed_file, os.path.join(POOL_DIR_S2_P2, "history_0.pt"))
print(f"🔥 Stage 2 Phase 2 seeded from Stage 2 Phase 1: {seed_file}")


# ── Environment Factory (Heterogeneous 2v2 & 2v3 Curriculum) ──
def make_s2_p2_env(env_rank: int):
  def _thunk():
    torch.set_num_threads(1)

    # 12 envs train 2v2 (40% Heuristic), 12 envs train 2v3 under-manned (40% Heuristic)
    opp_team_size = 2 if env_rank < 12 else 3

    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S2_P2,
        team="blue",
        device="cpu",
        p_random=0.05,     # 5% Random total
        p_heuristic=0.95,  # 80% Heuristic total (40% 2v2, 40% 2v3)
        frame_stack=3,
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=opp_team_size,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
        opponent_stats=[
            (3200.0, 1200.0),  # Standard tier
            (3600.0, 1400.0),  # Buffed Tier 1
            (4000.0, 1600.0),  # Buffed Tier 2 (extreme speed/kick)
        ],
        frame_stack=3,
    )
    env.reset(seed=4000 + env_rank)
    return env

  return _thunk


envs_s2_p2 = gym.vector.AsyncVectorEnv(
    [make_s2_p2_env(i) for i in range(NUM_ENVS)],
    context="fork",
    shared_memory=False,
)

# ── Model Initialization & Selective Warmstart ──
model_s2_p2 = TransformerActorCritic().to(device)
ckpt = torch.load(seed_file, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s2_p2.load_state_dict(state_dict, strict=True)
print("✅ Weights successfully loaded into Stage 2 Phase 2 model.")

# ── Training Loop ──
train_mappo(
    envs=envs_s2_p2,
    model=model_s2_p2,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,  # Official evaluation remains 2v2 regulation
    total_timesteps=15_000_000,
    num_envs=NUM_ENVS,
    num_steps=512,            # 24 envs * 2 agents * 512 = 24,576 batch size
    update_epochs=3,
    minibatch_size=2048,
    lr_init=3e-5,
    lr_final=2e-6,
    ent_coef_init=0.008,
    ent_coef_final=0.001,
    gamma=0.995,
    gae_lambda=0.96,
    active_tiers=["heuristic"],
    target_tier="heuristic",
    eval_episodes=50,
    eval_freq=250_000,
    save_dir=SAVE_DIR_S2_P2,
    pool_dir=POOL_DIR_S2_P2,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s2_p2.close()

Using device: cuda
🔥 Stage 2 Phase 2 seeded from Stage 2 Phase 1: models/stage2/phase2/best_model.pt
✅ Weights successfully loaded into Stage 2 Phase 2 model.
🚀 Entity-Transformer MAPPO Initialized | Format: 2v2 | Envs: 24 | Batch: 24576 | Device: cuda

📊 [EVALUATION @ Step 270,336 | Rollout SPS: 4116 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  90.0% | Reward: +4.064 | Goals: 162 Scored, 15 Conceded (+147 Net)
🏆 NEW CHAMPION REGISTERED @ step 270,336 -> models/stage2/phase2/pool/history_270336.pt
   ⭐⭐ PROMOTED! New Best Score (heuristic) -> [WR: 90.0%, Reward: +4.064, Net: +147]
      (Defeated previous record: [WR: None, Reward: None]) -> Saved: models/stage2/phase2/best_model.pt (Eval took 17.7s)

📊 [EVALUATION @ Step 516,096 | Rollout SPS: 4267 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  96.0% | Reward: +4.240 | Goals: 160 Scored, 10 Conceded (+150 Net)
🏆 NEW CHAMPION REGISTERED @ step 516,096 -> models/stage2/phase2/pool/history_516096.pt
   ⭐⭐ PROM

In [10]:
import os
import torch
import sys
# Ensure repository root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.visualization import evaluate_and_generate_html



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

html_path = evaluate_and_generate_html(
    red_agent="models/stage2/phase2/best_model.pt",
    blue_agent="models/stage2/phase2/final_model.pt",
    red_team_size=2,
    blue_team_size=2,
    device=device,
    filename="stage2_phase2_selfplay.html",
    num_episodes=10,
    max_steps=3600,
    action_repeat=10,
    pitch_width=1200.0,
    pitch_height=800.0,
    goal_height=220.0,
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/Notebooks/training_transformer/render/stage2_phase2_selfplay.html


# Phase 3

In [ ]:
import os
import shutil
import sys
import gymnasium as gym
import torch

# Ensure repository root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.env_adapter import MatchEnv
from src.rl_transformer.pool import PoolOpponentController
from src.rl_transformer.ppo import train_mappo
from src.rl_transformer.transformer_model import TransformerActorCritic

# ── Hardware Performance Flags ──
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── 2v2 Regulation Dimensions & Physics ──
TEAM_SIZE = 2
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS = 3600
ACTION_REPEAT = 6
NUM_ENVS = 24

# ── Directories & Seed Checkpoints ──
SAVE_DIR_S2_P2 = "models/stage2/phase2"
SAVE_DIR_S2_P3 = "models/stage2/phase3"
POOL_DIR_S2_P3 = os.path.join(SAVE_DIR_S2_P3, "pool")

os.makedirs(SAVE_DIR_S2_P3, exist_ok=True)
os.makedirs(POOL_DIR_S2_P3, exist_ok=True)

seed_file = os.path.join(SAVE_DIR_S2_P2, "final_model.pt")
if not os.path.exists(seed_file):
  seed_file = os.path.join(SAVE_DIR_S2_P2, "best_model.pt")

if not os.path.exists(seed_file):
  raise FileNotFoundError(f"Missing Stage 2 Phase 2 checkpoint to seed Phase 3: {seed_file}")

shutil.copy(seed_file, os.path.join(POOL_DIR_S2_P3, "champion.pt"))
shutil.copy(seed_file, os.path.join(POOL_DIR_S2_P3, "history_0.pt"))
print(f"🔥 Stage 2 Phase 3 seeded from Stage 2 Phase 2: {seed_file}")


# ── Environment Factory (Split 2v3 Handicap & 2v2 Self-Play) ──
def make_s2_p3_env(env_rank: int):
  def _thunk():
    torch.set_num_threads(1)

    # Envs 16-23 (33.3% of data): 2v3 Heuristic ONLY
    # Envs 0-15  (66.7% of data): 2v2 Self-Play (92.5%) + Random (7.5%)
    is_handicap_env = env_rank >= 16

    if is_handicap_env:
      opp_team_size = 3
      p_random = 0.0
      p_heuristic = 1.0   # 100% heuristic in these 8 envs
    else:
      opp_team_size = 2
      p_random = 0.075    # 0.075 * (16/24) = 5.0% total batch random
      p_heuristic = 0.0   # Remaining 92.5% of 16 envs = 61.7% total 2v2 self-play

    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S2_P3,
        team="blue",
        device="cpu",
        p_random=p_random,
        p_heuristic=p_heuristic,
        frame_stack=3,
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=opp_team_size,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
        opponent_stats=[
            (3200.0, 1200.0),  # Standard tier
            (3600.0, 1400.0),  # Buffed Tier 1
            (4000.0, 1600.0),  # Buffed Tier 2
        ],
        frame_stack=3,
    )
    env.reset(seed=6000 + env_rank)
    return env

  return _thunk


envs_s2_p3 = gym.vector.AsyncVectorEnv(
    [make_s2_p3_env(i) for i in range(NUM_ENVS)],
    context="fork",
    shared_memory=False,
)

# ── Model Initialization & Warmstart ──
model_s2_p3 = TransformerActorCritic().to(device)
ckpt = torch.load(seed_file, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s2_p3.load_state_dict(state_dict, strict=True)
print("✅ Weights successfully loaded into Stage 2 Phase 3 model.")

# ── 2v2 Competitive Self-Play Training Loop ──
train_mappo(
    envs=envs_s2_p3,
    model=model_s2_p3,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,               # Official promotion benchmark remains regulation 2v2
    total_timesteps=50_000_000,
    num_envs=NUM_ENVS,
    num_steps=512,                         # 24 envs * 2 agents * 512 = 24,576 batch size
    update_epochs=3,
    minibatch_size=2048,
    lr_init=2.5e-5,
    lr_final=2e-6,
    ent_coef_init=0.008,
    ent_coef_final=0.001,
    gamma=0.996,
    gae_lambda=0.96,
    active_tiers=["heuristic", "champion"],
    target_tier="champion",
    filter_thresholds={"heuristic": 0.85}, # Defends against heuristic regress without over-blocking
    tier_ratios={"heuristic": 0.50, "champion": 0.50},
    eval_episodes=100,
    eval_freq=250_000,
    save_dir=SAVE_DIR_S2_P3,
    pool_dir=POOL_DIR_S2_P3,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s2_p3.close()

Using device: cuda
🔥 Stage 2 Phase 3 seeded from Stage 2 Phase 2: models/stage2/phase2/final_model.pt
✅ Weights successfully loaded into Stage 2 Phase 3 model.
🚀 Entity-Transformer MAPPO Initialized | Format: 2v2 | Envs: 24 | Batch: 24576 | Device: cuda

📊 [EVALUATION @ Step 270,336 | Rollout SPS: 3103 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  92.0% | Reward: +4.906 | Goals: 193 Scored, 10 Conceded (+183 Net)
   ⚔️  vs Champion  [TARGET] | WR:  38.0% | Reward: +0.202 | Goals: 34 Scored, 23 Conceded (+11 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: 38.0%, Reward: +0.202, Net: +11] (Eval took 40.3s)

📊 [EVALUATION @ Step 516,096 | Rollout SPS: 3026 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  96.0% | Reward: +4.460 | Goals: 169 Scored, 9 Conceded (+160 Net)
   ⚔️  vs Champion  [TARGET] | WR:  56.0% | Reward: +0.916 | Goals: 42 Scored, 14 Conceded (+28 Net)
🏆 NEW CHAMPION REGISTERED @ step 516,096 -